# Create the ClickPipe through the API

The order of the workshop's setup is **migrate, create the ClickPipe, then seed**:

1. apply the Postgres migrations, so the six tables exist (and are empty);
2. create the **Postgres CDC ClickPipe** that replicates them into `retail_oltp`;
3. seed Postgres: the rows reach ClickHouse as changes, through the pipe.

The notebook makes the same calls as `make clickpipes`; both use `tools/clickpipes.py`. It reads your `.env` (nothing secret is printed) and needs an **API key**: in the ClickHouse Cloud console, *Organization, API keys*, create a key with the Admin role, and set `CLICKHOUSE_CLOUD_API_KEY_ID` and `CLICKHOUSE_CLOUD_API_KEY_SECRET` in `.env`.

To run it: `make notebook`, then open <http://localhost:8888>.

In [ ]:
import json
import os
import sys
from pathlib import Path

# The notebook lives in notebooks/; paths such as PGSSLROOTCERT are relative to the repo root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "tools"))


def load_env(path=ROOT / ".env"):
    """Read .env as written (no quotes, no inline comments) without printing any value."""
    for line in path.read_text().splitlines():
        line = line.rstrip("\r")
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ.setdefault(key, value)


load_env()
print("loaded .env, keys:", sorted(k for k in os.environ if k.startswith(("CLICKHOUSE_", "PG"))))

## 1. Apply the Postgres migrations

This creates the `clickshop` database if it is missing and applies `sql/postgres/*.sql`. **Nothing is loaded yet**: the tables are empty.

In [ ]:
import seed_postgres

seed_postgres.run(schema_only=True)

## 2. Find your ClickHouse service

The Cloud API is told apart by organisation and service. The service is the one whose host is your `CLICKHOUSE_HOST`.

In [ ]:
from cloud_api import context_from_env

api, ctx = context_from_env()
print(f"organization {ctx.organization_id}")
print(f"service      {ctx.service_name} ({ctx.service_id})")

## 3. The request

This is the whole ClickPipe, as the API wants it. Each part is a choice you make in the console in module 03:

| In the request | In the console |
|---|---|
| `source.postgres.host`, `port`, `database`, `credentials` | the source connection of your Postgres instance |
| `tlsHost`, `caCertificate` | the certificate that the pipe verifies (the file `PGSSLROOTCERT` points at) |
| `settings.replicationMode: "cdc"` | Replication method **Initial load + CDC** |
| `settings.syncIntervalSeconds: 60` | **Sync interval (seconds)** |
| `settings.allowNullableColumns: false` | **Preserve NULL values from source**, off |
| `settings.deleteOnMerge: false` | **Remove deleted rows during merges**, off |
| `tableMappings` | the six tables you select, each kept under its own name (no schema prefix) |
| `destination.database` | **Destination database**: `retail_oltp` |

The password and the certificate are masked in the output.

In [ ]:
import clickpipes

payload = clickpipes.payload_from_env()
print(json.dumps(clickpipes.redacted(payload), indent=2))

## 4. Create the pipe

`create_pipe` posts the request, unless the pipe already exists, by that name or made in the console with the same source and destination (then it does nothing, so you can run the cell again). The pipe goes through **Provisioning, Setup, Snapshot** and ends **Running**.

In [ ]:
pipe, created = clickpipes.create_pipe(api, ctx, payload)
print(("created" if created else "already there:"), pipe["name"], pipe["id"], "state", pipe.get("state"))
clickpipes.wait_until_running(api, ctx, pipe["id"])

The pipe has created its tables in `retail_oltp`, still empty:

In [ ]:
from connections import ch_client

ch = ch_client()
for name, engine in ch.query("SELECT name, engine FROM system.tables WHERE database = 'retail_oltp' ORDER BY name").result_rows:
    print(f"{name:55s} {engine}")

## 5. Seed Postgres

Now load the catalogue, customers, stock and order history. The ClickPipe is already running, so they arrive in ClickHouse as changes, within about one sync interval.

In [ ]:
seed_postgres.run()

In [ ]:
# Wait until ClickHouse holds what Postgres holds (about a minute with a 60 s sync interval).
clickpipes.wait_for_mirror(timeout=600)

from nuke_data import clickhouse_counts, postgres_counts
tables = list(clickpipes.TABLES)
print("Postgres  ", postgres_counts(tables))
print("ClickHouse", clickhouse_counts(ch, tables))

## Next

Build the search table and the real-time rollups: `make ch-search` and `make ch-mvs`. To start over, delete the pipe from the console, or run the cell below.

In [ ]:
# Uncomment to delete the pipe again.
# api.request("DELETE", f"{clickpipes.pipes_path(ctx)}/{pipe['id']}")